In [1]:
!pip install pymupdf

In [2]:
from google.colab import files

uploaded = files.upload()

ModuleNotFoundError: No module named 'google.colab'

In [3]:
# For local testing without Colab, you can use this code to read a file path from the user:
"""
source_path = input("Paste the full file path: ").strip()
file_name = source_path.split("/")[-1]

with open(source_path, "rb") as f:
    file_bytes = f.read()

uploaded = {file_name: file_bytes}

with open(file_name, "wb") as f:
    f.write(file_bytes)
"""

In [6]:
import fitz  # PyMuPDF
import re
from collections import Counter

# ---------- CLEANING ----------
def normalize_text(text):
    text = text.replace("\xa0", " ")
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

# ---------- HEADER/FOOTER REMOVAL ----------
def clean_candidate_line(line):
    line = line.strip()
    line = re.sub(r'Page\s+\d+\s+of\s+\d+', 'Page X of Y', line, flags=re.IGNORECASE)
    return line

def find_repeated_lines(page_texts, min_repeats=2):
    counter = Counter()

    for text in page_texts:
        lines = [l.strip() for l in text.splitlines() if l.strip()]
        candidates = lines[:5] + lines[-5:]  # check more top/bottom lines

        # use set so same line on one page only counts once
        cleaned_candidates = set(clean_candidate_line(line) for line in candidates)

        for line in cleaned_candidates:
            counter[line] += 1

    return {line for line, count in counter.items() if count >= min_repeats}

def remove_repeated_lines(text, repeated):
    cleaned_lines = []
    for line in text.splitlines():
        stripped = line.strip()
        normalized = clean_candidate_line(stripped)
        if normalized not in repeated:
            cleaned_lines.append(line)
    return "\n".join(cleaned_lines)

# ---------- MAIN PARSER ----------
def parse_pdf(file_path):
    doc = fitz.open(file_path)
    page_texts = []

    for page in doc:
        text = page.get_text("text")
        page_texts.append(text)

    doc.close()

    repeated_lines = find_repeated_lines(page_texts)
    cleaned_pages = [remove_repeated_lines(p, repeated_lines) for p in page_texts]

    full_text = "\n\n".join(cleaned_pages)
    return normalize_text(full_text)

In [7]:
parsed_results = {}

for file_name in uploaded.keys():
    print(f"\nProcessing: {file_name}")

    text = parse_pdf(file_name)
    parsed_results[file_name] = text

    print("Preview:\n")
    print(text[:1000])  # first 1000 characters
    print("\nLength:", len(text))
    print("="*80)


Processing: dhs_form_11000-6.pdf
Preview:

DEPARTMENT OF HOMELAND SECURITY 
NON-DISCLOSURE AGREEMENT
Protected Critical Infrastructure Information (PCII)
Agreement in consideration of my being granted conditional access to certain information, specified below, that is owned 
by, produced by, or in the possession of the United States Government.
(Signer will acknowledge the category or categories of information that he or she may have access to, and the signer's willingness to 
comply with the standards for protection by placing his or her initials in front of the applicable category or categories.)
Initials:
I attest that I am familiar with, and I will comply with all requirements of the PCII program set out in the Critical 
Infrastructure Information Act of 2002 (CII Act) (Title II, Subtitle B, of the Homeland Security Act of 2002, Public Law 
107-296, 196 Stat. 2135, 6 USC 101 et seq.), as amended, the implementing regulations thereto (6 CFR Part 29), as 
amended, and the applicable

In [8]:
for file_name, text in parsed_results.items():
    print(f"\nFile: {file_name}")
    print(f"Character count: {len(text)}")
    print(f"Word count: {len(text.split())}")
    print("-"*50)


File: dhs_form_11000-6.pdf
Character count: 12472
Word count: 1928
--------------------------------------------------


In [9]:
import os

os.makedirs("parsed_outputs", exist_ok=True)

for file_name, text in parsed_results.items():
    output_name = f"parsed_outputs/{file_name.replace('.pdf', '.txt')}"

    with open(output_name, "w", encoding="utf-8") as f:
        f.write(text)

print("All files saved.")

All files saved.


In [11]:
import re

# ──────────────────────────────────────────────
# STEP 3 – Clause-Aware Chunking
# ──────────────────────────────────────────────

# --- Token counting (word-level approximation) ---
def count_tokens(text: str) -> int:
    """Approximate token count using whitespace split (≈ word count)."""
    return len(text.split())


# --- Clause / section boundary detection ---
CLAUSE_BOUNDARY_RE = re.compile(
    r"""
    (?:^|\n)          # start of string or newline
    (?:
        (?:Section|Article|Clause|Exhibit|Schedule|Annex|Addendum)\s+\d+(?:\.\d+)*  # Section 4.2 / Article 3
        | \d+(?:\.\d+)+\.?\s+[A-Z]                                                  # 1.2 Heading  /  1.2.3 Sub
        | \d+\.\s+[A-Z]                                                               # 1. Heading
        | [A-Z][A-Z\s]{3,}(?:\.|:)                                                    # ALL-CAPS HEADING.
    )
    """,
    re.VERBOSE | re.MULTILINE,
)

def find_clause_boundaries(text: str) -> list[int]:
    """Return sorted list of character positions where clause headers begin."""
    return sorted({m.start() for m in CLAUSE_BOUNDARY_RE.finditer(text)})


# --- Core chunking function ---
def chunk_document(
    text: str,
    chunk_size: int = 900,
    overlap: int = 175,
    min_tokens: int = 800,
    max_tokens: int = 1000,
) -> list[dict]:
    """
    Split *text* into clause-aware, overlapping chunks.

    Parameters
    ----------
    text       : full document string
    chunk_size : target token count per chunk  (default 900)
    overlap    : overlap token count with previous chunk (default 175)
    min_tokens : lower bound for validation logging (default 800)
    max_tokens : upper bound for validation logging (default 1000)

    Returns
    -------
    List of dicts with keys:
        chunk_index  – zero-based position
        text         – chunk content
        token_count  – approximate token count
        start_char   – character offset in original text (start)
        end_char     – character offset in original text (end, exclusive)
        out_of_range – True if token_count is outside [min_tokens, max_tokens]
    """
    words = text.split()
    # Build a word-index → char-offset mapping so we can report start/end_char
    # We scan the text once to record where each word begins.
    char_offsets: list[int] = []
    pos = 0
    for word in words:
        pos = text.index(word, pos)
        char_offsets.append(pos)
        pos += len(word)

    clause_char_positions = set(find_clause_boundaries(text))

    def nearest_clause_boundary_before(word_idx: int) -> int:
        """Return the largest word index ≤ word_idx that sits on a clause boundary."""
        if not clause_char_positions or word_idx == 0:
            return word_idx
        char = char_offsets[word_idx] if word_idx < len(char_offsets) else char_offsets[-1]
        # Walk backwards from word_idx looking for a word that starts on a boundary
        for wi in range(word_idx, max(word_idx - 50, -1), -1):  # look back ≤50 words
            if char_offsets[wi] in clause_char_positions:
                return wi
        return word_idx

    chunks: list[dict] = []
    i = 0
    chunk_index = 0
    step = chunk_size - overlap          # advance by non-overlapping portion

    while i < len(words):
        end_word = min(i + chunk_size, len(words))

        # Prefer to end at a clause boundary (snap end_word forward or back
        # by up to 10% of chunk_size to land on a natural break).
        tolerance = max(1, chunk_size // 10)
        best_end = end_word
        if end_word < len(words):
            for wi in range(end_word, min(end_word + tolerance, len(words))):
                if char_offsets[wi] in clause_char_positions:
                    best_end = wi
                    break
            else:
                for wi in range(end_word - 1, max(end_word - tolerance, i), -1):
                    if char_offsets[wi] in clause_char_positions:
                        best_end = wi
                        break
            end_word = best_end

        chunk_words = words[i:end_word]
        chunk_text_str = " ".join(chunk_words)
        token_count = len(chunk_words)

        start_char = char_offsets[i]
        end_char = (
            char_offsets[end_word - 1] + len(words[end_word - 1])
            if end_word > i else start_char
        )

        out_of_range = not (min_tokens <= token_count <= max_tokens)

        chunks.append({
            "chunk_index": chunk_index,
            "text": chunk_text_str,
            "token_count": token_count,
            "start_char": start_char,
            "end_char": end_char,
            "out_of_range": out_of_range,
        })

        i += step
        chunk_index += 1

        if end_word >= len(words):
            break

    return chunks


# --- Validation / reporting helper ---
def validate_chunks(chunks: list[dict], file_label: str = "") -> None:
    """Print a summary table and flag out-of-range chunks."""
    label = f" [{file_label}]" if file_label else ""
    print(f"\n{'='*70}")
    print(f"Chunk Validation Report{label}")
    print(f"{'='*70}")
    print(f"{'#':>4}  {'Tokens':>7}  {'Start':>8}  {'End':>8}  {'Status'}")
    print(f"{'-'*4}  {'-'*7}  {'-'*8}  {'-'*8}  {'-'*15}")

    flagged = []
    for c in chunks:
        status = "⚠ OUT OF RANGE" if c["out_of_range"] else "✓ OK"
        print(f"{c['chunk_index']:>4}  {c['token_count']:>7}  "
              f"{c['start_char']:>8}  {c['end_char']:>8}  {status}")
        if c["out_of_range"]:
            flagged.append(c["chunk_index"])

    print(f"\nTotal chunks : {len(chunks)}")
    print(f"Flagged chunks (outside 800-1000 tokens): {len(flagged)}"
          + (f"  → indices {flagged}" if flagged else ""))
    print(f"{'='*70}\n")


print("  Step 3 – Chunking helpers defined.")
print("    Functions available: count_tokens, chunk_document, validate_chunks")


  Step 3 – Chunking helpers defined.
    Functions available: count_tokens, chunk_document, validate_chunks


In [12]:
# ──────────────────────────────────────────────
# STEP 3 – Run chunking on all parsed documents
# ──────────────────────────────────────────────

all_chunks: dict[str, list[dict]] = {}

for file_name, text in parsed_results.items():
    print(f"\nChunking: {file_name}")

    chunks = chunk_document(text)          # clause-aware, overlap=175, target=900
    all_chunks[file_name] = chunks

    # Validation table
    validate_chunks(chunks, file_label=file_name)

    # Preview first chunk
    print("First chunk preview:")
    print("-" * 60)
    print(chunks[0]["text"][:500])
    print("-" * 60)
    if len(chunks) > 1:
        # Verify overlap: count shared tokens between chunk 0 and chunk 1
        tokens_0 = set(chunks[0]["text"].split())
        tokens_1 = set(chunks[1]["text"].split())
        shared = len(tokens_0 & tokens_1)
        print(f"Overlap sanity check (unique tokens in common, chunks 0→1): {shared}")
    print()



Chunking: dhs_form_11000-6.pdf

Chunk Validation Report [dhs_form_11000-6.pdf]
   #   Tokens     Start       End  Status
----  -------  --------  --------  ---------------
   0      900         0      5842  ✓ OK
   1      900      4744     10425  ✓ OK
   2      478      9284     12472  ⚠ OUT OF RANGE

Total chunks : 3
Flagged chunks (outside 800-1000 tokens): 1  → indices [2]

First chunk preview:
------------------------------------------------------------
DEPARTMENT OF HOMELAND SECURITY NON-DISCLOSURE AGREEMENT Protected Critical Infrastructure Information (PCII) Agreement in consideration of my being granted conditional access to certain information, specified below, that is owned by, produced by, or in the possession of the United States Government. (Signer will acknowledge the category or categories of information that he or she may have access to, and the signer's willingness to comply with the standards for protection by placing his or her i
------------------------------------

STEP 4


In [13]:
# ──────────────────────────────────────────────
# STEP 4 – Tag Clause Metadata
# ──────────────────────────────────────────────

import re
import json

CLAUSE_PATTERNS = {
    "Payment / financial terms": [
        r"\bpayment\b", r"\bpayments\b", r"\bpay\b", r"\bfee\b", r"\bfees\b",
        r"\bcompensation\b", r"\binvoice\b", r"\binvoices\b", r"\bbilling\b",
        r"\brate\b", r"\brates\b", r"\bprice\b", r"\bpricing\b",
        r"\breimbursement\b", r"\breimburse\b", r"\bexpense\b", r"\bexpenses\b",
        r"\bdue\b", r"\bdeposit\b"
    ],
    "Termination / cancellation": [
        r"\bterminate\b", r"\btermination\b", r"\bcancel\b", r"\bcancellation\b",
        r"\bexpired\b", r"\bexpiration\b", r"\bnotice\b", r"\bbreach\b"
    ],
    "Liability / indemnification": [
        r"\bliability\b", r"\bliable\b", r"\bindemnify\b", r"\bindemnification\b",
        r"\bhold harmless\b", r"\bdamages\b", r"\blosses\b", r"\bclaims\b"
    ],
    "IP ownership / licensing": [
        r"\bintellectual property\b", r"\bownership\b", r"\bown\b",
        r"\blicense\b", r"\blicensing\b", r"\bcopyright\b",
        r"\btrademark\b", r"\bpatent\b", r"\bwork product\b"
    ],
    "Confidentiality / non-disclosure": [
        r"\bconfidential\b", r"\bconfidentiality\b", r"\bnon[- ]disclosure\b",
        r"\bnda\b", r"\bproprietary information\b", r"\bdisclosure\b"
    ],
    "Governing law / jurisdiction": [
        r"\bgoverning law\b", r"\bjurisdiction\b", r"\bvenue\b",
        r"\blaws of\b", r"\bcourts of\b"
    ],
    "Dispute resolution / arbitration": [
        r"\bdispute\b", r"\bdisputes\b", r"\barbitration\b",
        r"\bmediate\b", r"\bmediation\b", r"\bresolution\b"
    ],
    "Warranties / representations": [
        r"\bwarranty\b", r"\bwarranties\b", r"\brepresentation\b",
        r"\brepresentations\b", r"\bguarantee\b", r"\bguarantees\b",
        r"\bas is\b"
    ]
}

def tag_clause_metadata(chunk_text):
    tags = []
    text = chunk_text.lower()

    for clause_type, patterns in CLAUSE_PATTERNS.items():
        for pattern in patterns:
            if re.search(pattern, text, flags=re.IGNORECASE):
                tags.append(clause_type)
                break

    return tags

tagged_chunks = []

for file_name, chunks in all_chunks.items():
    for chunk in chunks:
        chunk_obj = {
            "chunk_text": chunk["text"],
            "chunk_index": chunk["chunk_index"],
            "source_document": file_name,
            "clause_tags": tag_clause_metadata(chunk["text"]),
            "token_count": chunk["token_count"]
        }
        tagged_chunks.append(chunk_obj)

print(f"Step 4 complete. Total tagged chunks: {len(tagged_chunks)}")

print("\nPreview of first 2 tagged chunks:\n")
for item in tagged_chunks[:2]:
    print(json.dumps(item, indent=2))
    print("-" * 80)

Step 4 complete. Total tagged chunks: 3

Preview of first 2 tagged chunks:

{
  "chunk_text": "DEPARTMENT OF HOMELAND SECURITY NON-DISCLOSURE AGREEMENT Protected Critical Infrastructure Information (PCII) Agreement in consideration of my being granted conditional access to certain information, specified below, that is owned by, produced by, or in the possession of the United States Government. (Signer will acknowledge the category or categories of information that he or she may have access to, and the signer's willingness to comply with the standards for protection by placing his or her initials in front of the applicable category or categories.) Initials: I attest that I am familiar with, and I will comply with all requirements of the PCII program set out in the Critical Infrastructure Information Act of 2002 (CII Act) (Title II, Subtitle B, of the Homeland Security Act of 2002, Public Law 107-296, 196 Stat. 2135, 6 USC 101 et seq.), as amended, the implementing regulations thereto (6